In [7]:
%pip install langchain langchain-community 
%pip install -U langchain-ollama

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
! ollama ls

NAME                       ID              SIZE      MODIFIED      
llama3.1:8b                46e0c10c039e    4.9 GB    7 days ago       
nomic-embed-text:latest    0a109f422b47    274 MB    7 days ago       
deepseek-r1:latest         0a8c26691023    4.7 GB    12 months ago    


In [8]:
# basic llm test
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model='llama3.1:8b'
)
print(llm.invoke("Explain AI in one sentence"))

content='Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as learning, problem-solving, decision-making, and perception.' additional_kwargs={} response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-03-31T11:48:10.0684869Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11298132000, 'load_duration': 121592300, 'prompt_eval_count': 16, 'prompt_eval_duration': 298594400, 'eval_count': 38, 'eval_duration': 10692679300, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'} id='lc_run--019d43b8-d0ac-7af1-b0ba-a82d8586ea75-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 16, 'output_tokens': 38, 'total_tokens': 54}


In [10]:
# chain implementation

from langchain_core.prompts import PromptTemplate
from langchain_ollama.chat_models import ChatOllama

llm = ChatOllama(
    model='llama3.1:8b'
)
prompt = PromptTemplate(
    input_variables=['topics'],
    template="Explain {topic} in simple words"
)

chain = llm | prompt
print(chain.invoke("Machine Learning"))

text='Explain content="Machine learning is a subset of artificial intelligence that involves training algorithms to learn from data and improve their performance on a task over time. Here are some key concepts related to machine learning:\\n\\n**Types of Machine Learning:**\\n\\n1. **Supervised Learning:** The algorithm is trained on labeled data, where the output variable is already known.\\n2. **Unsupervised Learning:** The algorithm is trained on unlabeled data, and it must find patterns or structure on its own.\\n3. **Reinforcement Learning:** The algorithm learns by interacting with an environment and receiving feedback in the form of rewards or penalties.\\n\\n**Machine Learning Algorithms:**\\n\\n1. **Linear Regression:** A linear model that predicts a continuous output variable based on one or more input features.\\n2. **Decision Trees:** A tree-like model that splits data into subsets based on feature values, used for classification and regression tasks.\\n3. **Random Forests:

In [13]:
# sequential chain

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = PromptTemplate.from_template("Explain {topic}")
chain1 = prompt1 | llm | StrOutputParser()

prompt2 = PromptTemplate.from_template("Summarize in one line: {text}. Only plaintext")
chain2 = prompt2 | llm | StrOutputParser()

overall_chain = {"text": chain1} | chain2
print(overall_chain.invoke({"topic": "Artificial Intelligence"}))

Artificial intelligence (AI) is a field of computer science that focuses on creating intelligent machines capable of performing tasks typically requiring human intelligence, such as learning, problem-solving, decision-making, and perception.


In [14]:
# Sequential chain (With Variables)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt1 = ChatPromptTemplate.from_template("Explain {topic}")
prompt2 = ChatPromptTemplate.from_template("Give 3 key points from: {explanation} Only plaintext")

# We use RunnablePassthrough.assign to keep 'explanation' in the dictionary 
chain = (
    {"topic": RunnablePassthrough()} 
    | RunnablePassthrough.assign(explanation=prompt1 | llm | StrOutputParser())
    | RunnablePassthrough.assign(points=prompt2 | llm | StrOutputParser())
)

result = chain.invoke("Neural Networks")
print(result)

{'topic': 'Neural Networks', 'explanation': 'Neural networks! A fundamental concept in machine learning and artificial intelligence.\n\n**What is a Neural Network?**\n\nA neural network (NN) is a computational model inspired by the structure and function of the human brain. It\'s composed of layers of interconnected nodes or "neurons" that process and transmit information.\n\nThink of a neural network as a complex system with multiple processing units working together to solve a problem, similar to how our brains work when we perceive, learn, and make decisions.\n\n**Key Components:**\n\n1. **Neurons (Nodes)**: These are the basic computing elements in a neural network. Each node processes a small amount of information from its inputs.\n2. **Connections**: Nodes are connected by directed links or edges, which allow them to exchange information with each other.\n3. **Activation Functions**: When a neuron receives multiple inputs, it applies an activation function to determine whether th

## Memory Component in LangChain (LCEL)

In [ ]:
# Memory Component
 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model='llama3.1:8b'
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

chain = prompt | llm
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

In [ ]:
config = {"configurable": {"session_id": "user_123"}}

# First interaction
response1 = with_message_history.invoke(
    {"question": "Hi, my name is Gemini."}, 
    config=config
)
# print("Store: ", store)

# Second interaction (It remembers the name)
response2 = with_message_history.invoke(
    {"question": "What is my name?"}, 
    config=config
)

print(response2.content)

Store:  {'user_123': InMemoryChatMessageHistory(messages=[HumanMessage(content='Hi, my name is Gemini.', additional_kwargs={}, response_metadata={}), AIMessage(content="Nice to meet you, Gemini! How can I assist you today? Do you have any questions or topics you'd like to discuss? I'm here to help with anything from recommendations to solving problems or just chatting about your interests. What's on your mind?", additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-04-01T11:50:35.5404929Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16793267500, 'load_duration': 122041500, 'prompt_eval_count': 28, 'prompt_eval_duration': 317440000, 'eval_count': 53, 'eval_duration': 16115826400, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019d48e1-4f7a-76e3-94e8-cf8535a2638b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 53, 'total_tokens': 81}), HumanMessage(content='W